In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import glob

from src.data.data_processing import Normalizer, load_sample
from src.models.fno_model import FNO3d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

time_labels = ['1 days', '2 days', '4 days', '7 days', '11 days', '17 days',
               '25 days', '37 days', '53 days', '77 days', '111 days', '158 days',
               '226 days', '323 days', '1.3 years', '1.8 years', '2.6 years',
               '3.6 years', '5.2 years', '7.3 years', '10.4 years', '14.8 years',
               '21.1 years', '30.0 years']


KeyboardInterrupt



In [ ]:
checkpoint_path = "checkpoints/fno_full_best.pt"
normalizer_path = "checkpoints/normalizer.pkl"

all_files = sorted(glob.glob("dataset/test_data/*.npz"))
if not all_files:
    raise RuntimeError("No test files found at dataset/test_data/*.npz")

test_files = all_files
normalizer = Normalizer.load(normalizer_path)

model = FNO3d(in_ch=11, width=48, modes_t=8, modes_z=12, modes_r=12).to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print("Model and normalizer loaded successfully.")

In [ ]:
sample_file = test_files[0]

x, gas_true, pressure_true = load_sample(sample_file, normalizer=normalizer, target_z=51)
x = x.unsqueeze(0).to(device)

with torch.no_grad():
    gas_pred, pressure_pred = model(x)

gas_pred = gas_pred.squeeze(0).squeeze(0).cpu().numpy()
pressure_pred = pressure_pred.squeeze(0).squeeze(0).cpu().numpy()
gas_true = gas_true.squeeze(0).numpy()
pressure_true = pressure_true.squeeze(0).numpy()

pressure_pred_phys = normalizer.denormalize("pressure_buildup", pressure_pred)
pressure_true_phys = normalizer.denormalize("pressure_buildup", pressure_true)

Loaded checkpoint: checkpoints/fno_coords_200_best.pt
Train files used for stats: 200
Validation files to inspect: 50


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
t_steps = [5, 12, 23] # Early, Middle, Final (30 years)

for i, t in enumerate(t_steps):
    # Ground Truth row
    im0 = axes[0, i].imshow(gas_true[t], origin="lower", aspect="auto", cmap="jet")
    axes[0, i].set_title(f"True Saturation: {time_labels[t]}")
    plt.colorbar(im0, ax=axes[0, i])
    
    # Prediction row
    im1 = axes[1, i].imshow(gas_pred[t], origin="lower", aspect="auto", cmap="jet")
    axes[1, i].set_title(f"Predicted Saturation: {time_labels[t]}")
    plt.colorbar(im1, ax=axes[1, i])

plt.suptitle("Spatio-Temporal Evolution: Ground Truth vs FNO3d Prediction")
plt.tight_layout()
plt.show()

--------------
Average validation set gas saturation R2 score is: 0.778949
Average validation set pressure buildup R2 score is: 0.837354
--------------
Average validation set gas saturation MAE is: 0.030107
Average validation set pressure buildup MAE is: 5.227349


In [ ]:
def get_r2(pred, target):
    target_mean = np.mean(target)
    ss_res = np.sum((target - pred) ** 2)
    ss_tot = np.sum((target - target_mean) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-8))

gas_r2 = get_r2(gas_pred, gas_true)
pres_r2 = get_r2(pressure_pred_phys, pressure_true_phys)

print(f"Results for file: {os.path.basename(sample_file)}")
print(f"---------------------------------------------")
print(f"Gas Saturation R2 Score: {gas_r2:.4f}")
print(f"Pressure Buildup R2 Score: {pres_r2:.4f}")

if gas_r2 > 0.95:
    print("Conclusion: Excellent performance (R2 > 0.95)")
else:
    print("Conclusion: Model captures trends but may need more training/modes.")

Sample path: dataset/train_data\data_1903.npz
Gas R2: 0.94071156
Pressure R2: 0.9829859
Gas MAE: 0.027298924
Pressure MAE: 3.34012
